# LSTM (Long Short-Term Memory)

## 1. Purpose
LSTM is a type of recurrent neural network designed to learn dependencies
across sequences and help reduce the vanishing-gradient problem.

It maintains two states:
- Hidden state (h_t)
- Cell state (c_t)

It uses three gates:
- Forget gate
- Input gate
- Output gate

The candidate cell state provides new information that can be added
to the cell state.

## 2. Input Preparation
For text, convert tokens into integer IDs and pass them through an embedding layer.

input_ids shape: [B, L]
embedded shape: [B, L, E]

B = Batch size
L = Sequence length
E = Embedding dimension
H = Hidden size
V = Vocabulary size
C = Number of classes

## 3. Model Architecture
Embedding -> LSTM -> Linear layer

Example:

embedded = self.embedding(input_ids)
output, (h_n, c_n) = self.lstm(embedded)
logits = self.fc(output)

For batch_first=True and a unidirectional LSTM:

input_ids: [B, L]
embedded:  [B, L, E]
output:    [B, L, H]
h_n:       [num_layers, B, H]
c_n:       [num_layers, B, H]

## 4. LSTM Outputs

output:
Contains the hidden state at every time step from the final LSTM layer.

h_n:
Contains the final hidden state of each LSTM layer.

c_n:
Contains the final cell state of each LSTM layer.

For a standard unidirectional LSTM:

output[:, -1, :] and h_n[-1] represent the same final-layer hidden
state when all sequences have the same valid length and the last position
is not padding.

The cell state c_n is not normally passed directly to the final
classification layer. It is available for further recurrent processing
or specialized architectures.

## 5. Logits
Logits are raw scores produced by the final linear layer.
They are not probabilities.

Convert the final hidden state into logits

For next-token prediction

### Sequence Classification
Use the final hidden state of the last LSTM layer.

final_hidden = h_n[-1]
logits = self.fc(final_hidden)

logits shape: [B, C]

### Next-Token Prediction
Use the hidden state at every time step.

logits = self.fc(output)

logits shape: [B, L, V]

The linear layer converts hidden dimension H into the number of classes C
or vocabulary size V.

Do not apply softmax before CrossEntropyLoss.

## 6. Loss Function

### Sequence Classification
logits shape: [B, C]
labels shape: [B]

criterion = nn.CrossEntropyLoss()
loss = criterion(logits, labels)

### Next-Token Prediction
Shift the input and target sequences by one token.

inputs = tokens[:, :-1]
targets = tokens[:, 1:]

logits shape: [B, L, V]
targets shape: [B, L]

Flatten the batch and sequence dimensions:

loss = criterion(
    logits.reshape(-1, logits.size(-1)),
    targets.reshape(-1)
)

Alternatively, move the vocabulary dimension:

loss = criterion(
    logits.transpose(1, 2),
    targets
)

For padded sequences:

criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

## 7. Optimizer
LSTM can use the same optimizers as CNNs and RNNs.

Examples:
- Adam
- AdamW
- SGD

Example:

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

Gradient clipping is often useful because recurrent networks can
experience exploding gradients.

loss.backward()

torch.nn.utils.clip_grad_norm_(
    model.parameters(),
    max_norm=1.0
)

optimizer.step()

Gradient clipping is optional and is not an optimizer.

## 8. Training Loop Changes
The basic training loop remains the same as CNN training.

Main differences:
- Prepare sequential inputs.
- Use an LSTM architecture.
- Unpack output and the tuple (h_n, c_n).
- Choose the final hidden state or all time-step outputs.
- Prepare targets according to the task.
- Calculate the loss using the correct tensor shapes.
- Consider gradient clipping.

## 9. Backpropagation
LSTM uses Backpropagation Through Time (BPTT).
PyTorch automatically computes gradients across recurrent time steps.

The cell state and gates help LSTM preserve and control information
over longer sequences.

## 10. Prediction
For sequence classification:

predictions = logits.argmax(dim=1)

For next-token prediction:

predictions = logits.argmax(dim=-1)

Sequence-classification predictions have shape [B].
Next-token predictions have shape [B, L].

### **Gradient Clipping handles exploding gradients but cant vanishing in rnn**

| LSTM component       | Analogy                                                                      |
| -------------------- | ---------------------------------------------------------------------------- |
| Cell state           | Information carried forward                                                  |
| Forget gate          | Decides how much existing information to retain                              |
| Input gate           | Controls how much new candidate information to add                           |
| Candidate cell state | New information that could be added                                          |
| Output gate          | Controls how much cell-state information is exposed through the hidden state |
| Hidden state         | Current exposed representation used for prediction and further processing    |


In [1]:
import torch
import torch.nn as nn

In [2]:
import torch
import torch.nn as nn

# 1. Define your dimensions
vocab_size = 1000      # Total number of unique words in your dictionary
embed_dim = 128        # Size of word embedding vectors (Hin)
hidden_size = 256      # Size of LSTM internal memory (Hout / Hcell)
batch_size = 8         # Number of sentences processed together (N)
seq_len = 5            # Length of each sentence (L)


In [3]:
# 2. Create a fake batch of input word IDs (e.g., 8 sentences, 5 words each)
# These represent your padded inputs from the DataLoader earlier!
sample_inputs = torch.randint(0, vocab_size, (batch_size, seq_len))  # ( LOW,HIGH,SHAPE WE NEED )
print("Input Token IDs Shape:", sample_inputs.shape)


Input Token IDs Shape: torch.Size([8, 5])


In [4]:
sample_inputs

tensor([[943,  98, 753, 277, 975],
        [280, 822, 156, 684, 557],
        [735, 343, 752, 373, 437],
        [286,  62, 748, 499,  39],
        [231, 503, 597, 790, 941],
        [582,  46, 772, 958, 867],
        [761,  74, 699, 609,  36],
        [894, 838, 482, 197, 880]])

In [5]:
# 3. Create an Embedding Layer to turn word IDs into feature vectors
embedding_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim)
embedded_inputs = embedding_layer(sample_inputs)
print("Embedded Input Shape (Hin):", embedded_inputs.shape) 
# Should be: torch.Size([8, 5, 128]) -> [Batch, Length, Embed_Dim]

Embedded Input Shape (Hin): torch.Size([8, 5, 128])


In [6]:
# 4. Create the PyTorch LSTM Layer
lstm_layer = nn.LSTM(
    input_size=embed_dim,
    hidden_size=hidden_size,
    num_layers=2,           # We are stacking 2 LSTM floors!
    batch_first=True        # Tells PyTorch our batch size comes first
)



In [7]:
# 5. Pass the embedded text through our LSTM
output, (h_n, c_n) = lstm_layer(embedded_inputs)

In [11]:
output.shape

torch.Size([8, 5, 256])

In [ ]:
print("\n--- LSTM OUTPUT SHAPES ---")
print("Main Output Shape (every word's state):", output.shape) 
# Expected: [8, 5, 256] -> [Batch, Length, Hidden_Size]

print("Final Hidden State (h_n) Shape:", h_n.shape) 
# Expected: [D * num_layers, Batch, Hidden_Size] -> [2, 8, 256]

print("Final Cell State (c_n) Shape:", c_n.shape) 
# Expected: [2, 8, 256] -> [2, 8, 256]


--- LSTM OUTPUT SHAPES ---
Main Output Shape (every word's state): torch.Size([8, 5, 256])
Final Hidden State (h_n) Shape: torch.Size([2, 8, 256])
Final Cell State (c_n) Shape: torch.Size([2, 8, 256])


TRAINING
Input → Prediction → Loss → Backpropagation → Parameter update

INFERENCE
Input → Prediction → Feed predicted token back → Next prediction

# **USING AN LSTM(Not own code under this)** 

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)

sentences = [
    "i love machine learning",
    "i love deep learning",
    "i love natural language processing",
    "machine learning is very useful",
    "deep learning is very powerful",
    "natural language processing is interesting",
    "i study artificial intelligence",
    "i study machine learning",
    "machine learning uses data",
    "deep learning uses neural networks",
    "neural networks learn patterns",
    "language models learn patterns",
    "i love artificial intelligence",
    "artificial intelligence is powerful",
    "machine learning is interesting",
    "i study deep learning",
    "neural networks use data",
    "language models generate text",
    "i learn neural networks",
    "deep learning learns patterns"
]

tokenized_sentences = [sentence.split() for sentence in sentences]

special_tokens = ["<PAD>", "<UNK>", "<EOS>"]

vocab = {token: idx for idx, token in enumerate(special_tokens)}

for sentence in tokenized_sentences:
    for word in sentence:
        if word not in vocab:
            vocab[word] = len(vocab)

id_to_word = {idx: word for word, idx in vocab.items()}

PAD_ID = vocab["<PAD>"]
UNK_ID = vocab["<UNK>"]
EOS_ID = vocab["<EOS>"]

vocab_size = len(vocab)

print("Vocabulary size:", vocab_size)
print("Vocabulary:", vocab)


def encode_sentence(sentence):
    tokens = sentence.split()
    token_ids = [vocab.get(token, UNK_ID) for token in tokens]
    token_ids.append(EOS_ID)
    return token_ids


class NextTokenDataset(Dataset):
    def __init__(self, sentences):
        self.samples = []

        for sentence in sentences:
            token_ids = encode_sentence(sentence)

            inputs = token_ids[:-1]
            targets = token_ids[1:]

            self.samples.append((inputs, targets))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        inputs, targets = self.samples[idx]
        return torch.tensor(inputs), torch.tensor(targets)


def collate_batch(batch):
    inputs, targets = zip(*batch)

    inputs = nn.utils.rnn.pad_sequence(
        inputs,
        batch_first=True,
        padding_value=PAD_ID
    )

    targets = nn.utils.rnn.pad_sequence(
        targets,
        batch_first=True,
        padding_value=PAD_ID
    )

    return inputs, targets


dataset = NextTokenDataset(sentences)

dataloader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_batch
)


class LSTMNextTokenModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim=64,
        hidden_size=128,
        num_layers=2
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=PAD_ID
        )

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        self.linear = nn.Linear(
            hidden_size,
            vocab_size
        )

    def forward(self, x):
        embedded = self.embedding(x)

        output, (h_n, c_n) = self.lstm(embedded)

        logits = self.linear(output)

        return logits


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = LSTMNextTokenModel(
    vocab_size=vocab_size,
    embed_dim=64,
    hidden_size=128,
    num_layers=2
).to(device)

criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_ID
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.003
)


first_inputs, first_targets = next(iter(dataloader))

first_inputs = first_inputs.to(device)
first_targets = first_targets.to(device)

with torch.no_grad():
    first_logits = model(first_inputs)

print("\n--- SHAPES ---")
print("Input token IDs:", first_inputs.shape)
print("Target token IDs:", first_targets.shape)
print("Logits:", first_logits.shape)

print("\n--- ONE EXAMPLE ---")
print("Input IDs:", first_inputs[0].tolist())
print("Target IDs:", first_targets[0].tolist())


for epoch in range(100):
    model.train()

    total_loss = 0.0
    total_tokens = 0

    for inputs, targets in dataloader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        logits = model(inputs)

        loss = criterion(
            logits.reshape(-1, vocab_size),
            targets.reshape(-1)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        valid_tokens = (targets != PAD_ID).sum().item()

        total_loss += loss.item() * valid_tokens
        total_tokens += valid_tokens

    average_loss = total_loss / total_tokens

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch + 1:3d}/100 | "
            f"Loss: {average_loss:.4f}"
        )


def predict_next_token(prompt):
    model.eval()

    token_ids = encode_sentence_without_eos(prompt)

    if not token_ids:
        raise ValueError("Prompt must contain at least one known token.")

    input_tensor = torch.tensor(
        [token_ids],
        dtype=torch.long,
        device=device
    )

    with torch.no_grad():
        logits = model(input_tensor)

        next_token_logits = logits[0, -1, :]

        next_token_id = torch.argmax(
            next_token_logits
        ).item()

    return id_to_word[next_token_id]


def encode_sentence_without_eos(sentence):
    return [
        vocab.get(word, UNK_ID)
        for word in sentence.lower().split()
    ]


def generate_text(prompt, max_new_tokens=5):
    model.eval()

    generated_ids = encode_sentence_without_eos(prompt)

    if not generated_ids:
        raise ValueError("Prompt must contain at least one token.")

    for _ in range(max_new_tokens):
        input_tensor = torch.tensor(
            [generated_ids],
            dtype=torch.long,
            device=device
        )

        with torch.no_grad():
            logits = model(input_tensor)

            next_token_logits = logits[0, -1, :].clone()

            next_token_logits[PAD_ID] = float("-inf")
            next_token_logits[UNK_ID] = float("-inf")

            next_token_id = torch.argmax(
                next_token_logits
            ).item()

        if next_token_id == EOS_ID:
            break

        generated_ids.append(next_token_id)

    return " ".join(
        id_to_word[token_id]
        for token_id in generated_ids
        if token_id != EOS_ID
    )


print("\n--- NEXT TOKEN PREDICTION ---")
print("Prompt: i love")
print("Prediction:", predict_next_token("i love"))

print("\n--- TEXT GENERATION ---")
print("Prompt: machine learning")
print("Generated:", generate_text("machine learning", max_new_tokens=10))


Vocabulary size: 30
Vocabulary: {'<PAD>': 0, '<UNK>': 1, '<EOS>': 2, 'i': 3, 'love': 4, 'machine': 5, 'learning': 6, 'deep': 7, 'natural': 8, 'language': 9, 'processing': 10, 'is': 11, 'very': 12, 'useful': 13, 'powerful': 14, 'interesting': 15, 'study': 16, 'artificial': 17, 'intelligence': 18, 'uses': 19, 'data': 20, 'neural': 21, 'networks': 22, 'learn': 23, 'patterns': 24, 'models': 25, 'use': 26, 'generate': 27, 'text': 28, 'learns': 29}

--- SHAPES ---
Input token IDs: torch.Size([4, 5])
Target token IDs: torch.Size([4, 5])
Logits: torch.Size([4, 5, 30])

--- ONE EXAMPLE ---
Input IDs: [9, 25, 27, 28, 0]
Target IDs: [25, 27, 28, 2, 0]
Epoch  10/100 | Loss: 0.9086
Epoch  20/100 | Loss: 0.3801
Epoch  30/100 | Loss: 0.3519
Epoch  40/100 | Loss: 0.3346
Epoch  50/100 | Loss: 0.3242
Epoch  60/100 | Loss: 0.3284
Epoch  70/100 | Loss: 0.3183
Epoch  80/100 | Loss: 0.3191
Epoch  90/100 | Loss: 0.3202
Epoch 100/100 | Loss: 0.3276

--- NEXT TOKEN PREDICTION ---
Prompt: i love
Prediction: nat